# Inflation curves (CPI / breakeven and real rates)

Deep-dive reference: **inflation-linked discounting** — dedicated `InflationCurve` when available, or a **nominal vs real** view via discount curves.

## Concept

**Inflation curves** describe expected **CPI** growth or **inflation swap** breakevens, used to price **TIPS**, **ILBs**, and **inflation swaps**. A common identity (Fisher-style, continuous-time shorthand) links **nominal** \(r_n\), **real** \(r_r\), and **inflation** \(\pi\): \(r_n \approx r_r + \pi\). When no dedicated curve type exists, you can still illustrate **real zero rates** from a **nominal** curve and an assumed **inflation** path.

## API walkthrough

Try **`InflationCurve`** from `finstack.core.market_data`. If it is missing, we **print** that and fall back to **`DiscountCurve`** for a **nominal** curve and a simple **real-rate** proxy.

In [ ]:
try:
    from finstack.core.market_data import InflationCurve  # type: ignore[attr-defined]
except ImportError:
    print(
        "InflationCurve is not bound in finstack.core.market_data for this build.\n"
        "Use core inflation types when they ship; below we proxy 'real' rates from nominal zeros.",
    )
else:
    print("InflationCurve is available — wire builder args per release docs.")
    print(InflationCurve)

## Practical example

**Nominal OIS** discount curve plus a **constant CPI breakeven** (illustrative) to derive **approximate real zeros** with \(r_r \approx r_n - \pi\).

**Caveats:** Real **CPI** is **lagged** (release delay and contract **observation lags**), **seasonal** (and often revised), and market breakevens embed an **inflation risk premium** and liquidity effects—so a flat \(\pi\) is only a teaching shortcut. In **continuous compounding**, if nominal and real zeros are defined consistently, \(DF(t) \approx e^{-r t}\) links discounting to zeros; a Fisher-style split \(r_n \approx r_r + \pi\) is still an approximation and must use compatible conventions in production.

In [ ]:
from datetime import date

from finstack.core.market_data import DiscountCurve

base = date(2024, 1, 1)
nominal = DiscountCurve(
    "USD-OIS-NOMINAL",
    base,
    [(0.0, 1.0), (1.0, 0.96), (2.0, 0.91), (5.0, 0.78), (10.0, 0.60)],
    day_count="act_365f",
)

pi = 0.025  # illustrative annualized expected inflation (not from a calibrated CPI curve)
print(f"Illustrative constant inflation pi = {pi:.4f} (~2.5% per year in this toy)")

for t in (1.0, 2.0, 5.0, 10.0):
    rn = nominal.zero(t)
    rr = rn - pi
    print(f"t={t}y  nominal_zero={rn:.6f}  approx_real_zero={rr:.6f}")

## Takeaways

- Prefer a dedicated **`InflationCurve`** (or CPI fixings + seasonality) when the bindings expose it.
- Until then, **`DiscountCurve`** gives **nominal** zeros; combine with **inflation assumptions** only for **illustration**, not for production ILB pricing.
- **Seasonality**, **lag structures** (e.g. 3M lag), and **instrument-specific** conventions matter for real inflation books.

## Calibration from inflation swap quotes

With the calibration engine, we can bootstrap an **inflation curve** from zero-coupon inflation swap (ZCIS) rates.  This requires:
1. A pre-existing **discount curve** (for PV calculations)
2. ZCIS quotes at various maturities with a **base CPI** level and **observation lag**

The engine solves for the CPI projection path that reprices each ZCIS to zero NPV.

In [ ]:
import json
from datetime import date

from finstack.core.market_data import DiscountCurve, MarketContext
from finstack.valuations import calibrate

def tenor(count, unit):
    return {"tenor": {"count": count, "unit": unit}}

# First build a discount curve as initial_market
dc = DiscountCurve(
    "USD-OIS", date(2025, 1, 15),
    [(0.0, 1.0), (1.0, 0.9478), (3.0, 0.8726), (5.0, 0.8099), (10.0, 0.6681)],
    day_count="act_365f",
)
initial_ctx = MarketContext()
initial_ctx.insert(dc)
initial_market_json = json.loads(initial_ctx.to_json())

plan = {
    "schema": "finstack.calibration/2",
    "plan": {
        "id": "us-cpi-curve",
        "description": "US CPI inflation curve from ZCIS quotes",
        "quote_sets": {
            "zcis": [
                {"class": "inflation", "inflation_swap": {
                    "maturity": "2026-01-15", "rate": 0.0250, "index": "USA-CPI-U", "convention": "USD-CPI"}},
                {"class": "inflation", "inflation_swap": {
                    "maturity": "2028-01-15", "rate": 0.0260, "index": "USA-CPI-U", "convention": "USD-CPI"}},
                {"class": "inflation", "inflation_swap": {
                    "maturity": "2030-01-15", "rate": 0.0255, "index": "USA-CPI-U", "convention": "USD-CPI"}},
                {"class": "inflation", "inflation_swap": {
                    "maturity": "2035-01-15", "rate": 0.0245, "index": "USA-CPI-U", "convention": "USD-CPI"}},
            ],
        },
        "steps": [{
            "id": "USD-CPI",
            "quote_set": "zcis",
            "kind": "inflation",
            "curve_id": "USD-CPI",
            "currency": "USD",
            "base_date": "2025-01-15",
            "discount_curve_id": "USD-OIS",
            "index": "USA-CPI-U",
            "observation_lag": "3M",
            "base_cpi": 315.5,
            "method": "Bootstrap",
            "interpolation": "linear",
        }],
        "settings": {"use_parallel": False},
    },
    "initial_market": initial_market_json,
}

result = calibrate(json.dumps(plan))
print("Success:", result.success)
print(result.report_to_dataframe().to_string(index=False))